# 手写二维注意力层（2D Attention）

## 核心思想

输入: `(num_rows, num_columns, hidden_size)` 的表格张量

拆成两步 attention:
1. **Cross-Column**: batch=R, seq_len=C, NO mask
2. **Cross-Row**: batch=C, seq_len=R, WITH mask

计算量从 O((RC)^2) 降到 O(RC^2 + CR^2)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Self-Attention (Multi-Head)

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.0):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)
        self.dropout = dropout

    def reshape_for_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, hidden_states, attention_mask=None):
        Q = self.reshape_for_heads(self.W_q(hidden_states))
        K = self.reshape_for_heads(self.W_k(hidden_states))
        V = self.reshape_for_heads(self.W_v(hidden_states))
        out = F.scaled_dot_product_attention(
            Q, K, V, attn_mask=attention_mask,
            dropout_p=self.dropout if self.training else 0.0, is_causal=False)
        B, h, T, d = out.shape
        return out.transpose(1, 2).contiguous().view(B, T, self.hidden_size)

In [3]:
# import math

# class SelfAttention(nn.Module):
#     def __init__(self, hidden_size, num_heads, dropout=0.0):
#         super().__init__()
#         assert hidden_size % num_heads == 0
#         self.hidden_size = hidden_size
#         self.num_heads = num_heads
#         self.d_k = hidden_size // num_heads

#         self.Wq = nn.Linear(hidden_size, hidden_size, bias=False)
#         self.Wk = nn.Linear(hidden_size, hidden_size, bias=False)
#         self.Wv = nn.Linear(hidden_size, hidden_size, bias=False)
#         self.Wo = nn.Linear(hidden_size, hidden_size, bias=False)
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, hidden_states, attention_mask=None):
#         B, T, _ = hidden_states.shape
#         Q = self.Wq(hidden_states)
#         K = self.Wk(hidden_states)
#         V = self.Wv(hidden_states)

#         Q = Q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
#         K = K.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
#         V = V.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

#         attn = torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(self.d_k)

#         if attention_mask is not None:
#             attn = attn.masked_fill(~attention_mask, float('-inf'))

#         attn_scores = F.softmax(attn, dim=-1)
#         attn_scores = self.dropout(attn_scores)

#         output = torch.matmul(attn_scores, V)
#         output = output.transpose(1, 2).contiguous().view(B, T, self.hidden_size)

#         return self.Wo(output)

## Attention Block (Attn + Residual + LayerNorm)

Pre-Norm 是先对输入做 LayerNorm 归一化，再送入子层（如 Attention 或 FFN）计算，最后通过残差连接将原始输入加回来，即 y = x + f(LN(x))。Post-Norm 则是先经过子层计算，再将结果与原始输入相加后做 LayerNorm 归一化，即 y = LN(f(x) + x)。

Pre-Norm 的优势在于训练更加稳定。由于残差连接中的恒等映射不经过 LayerNorm，梯度可以沿残差路径直接回传，不会被 LayerNorm 反复缩放。而 Post-Norm 中，梯度每经过一层都必须穿过 LayerNorm 的雅可比矩阵，在深层网络中这些缩放会逐层累积，容易导致梯度爆炸或消失，训练不稳定。因此，现代大语言模型（如 GPT-2、GPT-3、LLaMA）普遍采用 Pre-Norm 结构。

In [4]:
# PreNorm
# x → LayerNorm → Attention → dense → dropout → + x → 输出
#                                               ↑
#                                             残差（用原始 x）

# PostNorm
# Self-Attention → dense(输出投影) → Dropout → 残差 + LayerNorm
class AttentionBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.0):
        super().__init__()
        self.self_attn = SelfAttention(hidden_size, num_heads, dropout)
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden_states, attention_mask=None):
        # # PreNorm
        # hidden_states_normed=self.layer_norm(hidden_states)
        # res=self.dropout(self.dense(self.self_attn(hidden_states_normed,attention_mask)))
        # return hidden_states+res

        # PostNorm
        res=self.dropout(self.dense(self.self_attn(hidden_states,attention_mask)))
        return self.layer_norm(res+hidden_states)

## Transformer Layer (Attention + FFN)

In [5]:
class TransformerLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.0):
        super().__init__()
        self.attention = AttentionBlock(hidden_size, num_heads, dropout)
        self.ffn_up = nn.Linear(hidden_size, hidden_size * 4)
        self.ffn_down = nn.Linear(hidden_size * 4, hidden_size)
        self.ffn_norm = nn.LayerNorm(hidden_size)
        self.ffn_act = nn.GELU()

    def forward(self, hidden_states, attention_mask=None):
        x = self.attention(hidden_states, attention_mask)
        ffn_out = self.ffn_down(self.ffn_act(self.ffn_up(x)))
        return self.ffn_norm(ffn_out + x)

## 2D Attention Layer (核心)

In [6]:
class TwoDimensionalAttentionLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.0):
        super().__init__()
        self.cross_column_layer = TransformerLayer(hidden_size, num_heads, dropout)
        self.cross_row_layer = TransformerLayer(hidden_size, num_heads, dropout)

    def forward(self, hidden_states, attention_mask):
        # Step 1: Cross-Column Attention
        # (R, C, D) -> batch=R, seq_len=C, no mask
        h_out = self.cross_column_layer(hidden_states)

        # Step 2: Cross-Row Attention
        # transpose: (R, C, D) -> (C, R, D), batch=C, seq_len=R
        h_out = h_out.transpose(0, 1).contiguous()
        # mask: (R, R) -> (1, 1, R, R) broadcast to (C, h, R, R)
        mask_4d = attention_mask.unsqueeze(0).unsqueeze(0)
        v_out = self.cross_row_layer(h_out, mask_4d)

        # transpose back: (C, R, D) -> (R, C, D)
        return v_out.transpose(0, 1).contiguous()

## 测试

构造一个 4行6列 的表格，hidden_size=64, 8个注意力头

In [7]:
num_rows, num_cols, hidden_size, num_heads = 4, 6, 64, 8

# 构造输入
x = torch.randn(num_rows, num_cols, hidden_size)
print(f'输入 shape: {x.shape}')  # (4, 6, 64)

# 构造 attention_mask: Row0 是 query, Row1-3 是 context
# context 行不能看 query 行, query 行可以看所有行
mask = torch.zeros(num_rows, num_rows)
mask[1:, 0] = float('-inf')  # context rows cannot attend to query row
print(f'\nAttention Mask (0=attend, -inf=block):')
print(mask)

# 创建模型并前向传播
model = TwoDimensionalAttentionLayer(hidden_size, num_heads)
output = model(x, mask)
print(f'\n输出 shape: {output.shape}')  # (4, 6, 64)
print(f'输入输出 shape 一致: {x.shape == output.shape}')

输入 shape: torch.Size([4, 6, 64])

Attention Mask (0=attend, -inf=block):
tensor([[0., 0., 0., 0.],
        [-inf, 0., 0., 0.],
        [-inf, 0., 0., 0.],
        [-inf, 0., 0., 0.]])

输出 shape: torch.Size([4, 6, 64])
输入输出 shape 一致: True


## Shape 变化追踪

In [8]:
print('===== Shape 变化全流程 =====')
print(f'R={num_rows}, C={num_cols}, D={hidden_size}, h={num_heads}, d_k={hidden_size//num_heads}')
print()
print('--- Step 1: Cross-Column Attention ---')
print(f'  输入:          ({num_rows}, {num_cols}, {hidden_size})  [batch=R, seq=C]')
print(f'  Q/K/V 投影后:  ({num_rows}, {num_cols}, {hidden_size})')
print(f'  Q/K/V 拆多头:  ({num_rows}, {num_heads}, {num_cols}, {hidden_size//num_heads})')
print(f'  Attn 矩阵:    ({num_rows}, {num_heads}, {num_cols}, {num_cols})')
print(f'  输出:          ({num_rows}, {num_cols}, {hidden_size})')
print()
print('--- transpose(0,1) ---')
print(f'  shape 变为:    ({num_cols}, {num_rows}, {hidden_size})  [batch=C, seq=R]')
print()
print('--- Step 2: Cross-Row Attention ---')
print(f'  输入:          ({num_cols}, {num_rows}, {hidden_size})  [batch=C, seq=R]')
print(f'  Q/K/V 投影后:  ({num_cols}, {num_rows}, {hidden_size})')
print(f'  Q/K/V 拆多头:  ({num_cols}, {num_heads}, {num_rows}, {hidden_size//num_heads})')
print(f'  Attn 矩阵:    ({num_cols}, {num_heads}, {num_rows}, {num_rows})')
print(f'  Mask shape:    (1, 1, {num_rows}, {num_rows}) -> broadcast')
print(f'  输出:          ({num_cols}, {num_rows}, {hidden_size})')
print()
print('--- transpose(0,1) ---')
print(f'  最终输出:      ({num_rows}, {num_cols}, {hidden_size})')

===== Shape 变化全流程 =====
R=4, C=6, D=64, h=8, d_k=8

--- Step 1: Cross-Column Attention ---
  输入:          (4, 6, 64)  [batch=R, seq=C]
  Q/K/V 投影后:  (4, 6, 64)
  Q/K/V 拆多头:  (4, 8, 6, 8)
  Attn 矩阵:    (4, 8, 6, 6)
  输出:          (4, 6, 64)

--- transpose(0,1) ---
  shape 变为:    (6, 4, 64)  [batch=C, seq=R]

--- Step 2: Cross-Row Attention ---
  输入:          (6, 4, 64)  [batch=C, seq=R]
  Q/K/V 投影后:  (6, 4, 64)
  Q/K/V 拆多头:  (6, 8, 4, 8)
  Attn 矩阵:    (6, 8, 4, 4)
  Mask shape:    (1, 1, 4, 4) -> broadcast
  输出:          (6, 4, 64)

--- transpose(0,1) ---
  最终输出:      (4, 6, 64)


## 常见面试追问

| 问题 | 要点 |
|------|------|
| 为什么不直接做全局 attention? | 计算量 O((RC)^2) 太大，且丢失二维结构 |
| 两步的顺序重要吗? | 先列后行或先行后列都可以，但先列后行更常见 |
| 为什么 Cross-Column 不需要 mask? | 同行各列都是同一条记录的不同字段，应该互相可见 |
| 为什么 Cross-Row 需要 mask? | 防止 context 行看到 query 行（信息泄露） |
| mask 的 unsqueeze 是做什么? | (R,R)->(1,1,R,R)，第0维广播到batch=C，第1维广播到num_heads |
| 和 Axial Attention 的关系? | 本质相同，Axial Attention 也是沿不同轴分别做 attention |
| transpose(0,1) 的作用? | 交换 batch 和 seq_len 维度，复用同一套 MHA 代码实现不同方向 |